# CAMEL Ambient Solar Wind Validation

## Overview

This notebook demonstrates how to validate ambient solar wind models against in situ measurements near Earth. The workflow is based on the **CAMEL (Comprehensive Assessment of Models and Events using Library tools)** framework developed at the [Community Coordinated Modeling Center (CCMC)](https://ccmc.gsfc.nasa.gov/).

This effort is part of the international [ISWAT H1-01 Team](https://iswat-cospar.org/h1-01), which aims to establish a unified validation approach for solar wind models across the heliophysics and space weather communities.

The validation procedures presented in this notebook series are also available through an interactive web application built with Plotly Dash and Python. The **Ambient Solar Wind Validation Dashboard** can be accessed here:

- [CCMC CAMEL Ambient Solar Wind Validation Dashboard](https://ccmc.gsfc.nasa.gov/camel/AmbientSolarWind/home)

These notebooks provide a reproducible version of the same validation workflow and are intended to help users explore, reproduce, and extend the validation procedures of the CCMC CAMEL dashboard.

**Notebook version:** 1.0

---

## Notebook Scope

### 03 Binary Event Metrics

Binary event metrics evaluate whether a model correctly predicts the **occurrence or non-occurrence** of events defined by a user-selected threshold.

This notebook:
- loads the preprocessed example datasets generated in `01_api_and_preprocessing.ipynb`
- evaluates threshold-based binary event classifications for observed and modeled solar wind speed
- computes standard binary event metrics from the resulting contingency tables
- reproduces the binary-event visualization workflow used in the dashboard

---

## Expected Inputs

This notebook expects the following processed datasets:

```text
../example_data/processed/
├── omni_obs.csv
├── ambsowi_mod.csv
├── bravda_mod.csv
└── p27_mod.csv
```

## Imports

We import the metric and visual diagnostic functions from `utils/`. Note that these functions are an exact copy of the functions that are used in the CAMEL Ambient Solar Wind Dashboard. 


In [ ]:
import sys
from pathlib import Path

project_root = ".."
utils_path = "../utils"

if project_root not in sys.path:
    sys.path.append(project_root)
if utils_path not in sys.path:
    sys.path.append(utils_path)

import numpy as np
import pandas as pd
from IPython.display import display

from skill_scores import *
from utils.visualizations import *

## Load the processed example datasets

These CSV files are assumed to have already been preprocessed and aligned in `01_api_and_preprocessing.ipynb`. Note that if you want to change the selected time window for validation you need to go back to `01_api_and_preprocessing.ipynb`.


In [ ]:
data_dir = Path("../example_data/processed")

omni_obs = pd.read_csv(data_dir / "omni_obs.csv", parse_dates=["Time"], index_col="Time")
ambsowi_mod = pd.read_csv(data_dir / "ambsowi_mod.csv", parse_dates=["Time"], index_col="Time")
bravda_mod = pd.read_csv(data_dir / "bravda_mod.csv", parse_dates=["Time"], index_col="Time")
p27_mod = pd.read_csv(data_dir / "p27_mod.csv", parse_dates=["Time"], index_col="Time")

print("Observed shape:   ", omni_obs.shape)
print("AmbSoWi-ML shape: ", ambsowi_mod.shape)
print("BRaVDA shape:     ", bravda_mod.shape)
print("Persistence shape:", p27_mod.shape)


## Prepare inputs for the binary-event validation procedures

Convert the processed time series into the long-format structure expected by the CAMEL dashboard functions.

In [ ]:
def to_pred(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Model"})
    d["Name"] = name
    return d[["Time", "Model", "Name"]]

def to_obs(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Obs"})
    d["Name"] = name
    return d[["Time", "Obs", "Name"]]

observed_data = to_obs(omni_obs, "OMNI2 | Hourly averaged")

predicted_ambsowi = to_pred(ambsowi_mod, "AmbSoWi-ML")
predicted_bravda = to_pred(bravda_mod, "BRaVDA")
baseline_data = to_pred(p27_mod, "Persistence (27-days)")

## Compute binary classification metrics

Binary event metrics evaluate how well models can simulate the **occurrence or non-occurrence** of events defined by a threshold value.

Instead of comparing every point in time, this approach classifies outcomes for each time step into:
- **Hit**
- **Miss**
- **False Alarm**
- **Correct Rejection**

We compute a contingency table and derive some standard binary event metrics. To do so, we first need to define an **event threshold**.

In [ ]:
threshold = 450
print("Binary event threshold:", threshold)

In [ ]:
_, ct_ambsowi = classifyBinaryEvents(predicted_ambsowi, observed_data, threshold)

ambsowi_metrics = pd.DataFrame(
    {
        "Metric": [
            "True Positive Rate",
            "False Positive Rate",
            "Threat Score",
            "True Skill Statistic",
            "Bias",
        ],
        "Value": [
            truePositiveRate(ct_ambsowi)[0],
            falsePostiveRate(ct_ambsowi)[0],
            threatScore(ct_ambsowi)[0],
            trueSkillStatistics(ct_ambsowi)[0],
            bias(ct_ambsowi)[0],
        ],
    }
)

display(ct_ambsowi)
display(ambsowi_metrics.round(2))

## Compare binary classification metrics across all available models

This table extends the example above to all available series.

In [ ]:
def compute_binary_metric_table(models: dict, observed: pd.DataFrame, threshold: float) -> pd.DataFrame:
    rows = {}
    obs_long = to_obs(observed, "OMNI2 | Hourly averaged")
    for name, model_df in models.items():
        pred_long = to_pred(model_df, name)
        _, ct = classifyBinaryEvents(pred_long, obs_long, threshold)
        rows[name] = {
            "tp": int(ct["tp"].iloc[0]),
            "fp": int(ct["fp"].iloc[0]),
            "fn": int(ct["fn"].iloc[0]),
            "tn": int(ct["tn"].iloc[0]),
            "TPR": float(truePositiveRate(ct)[0]),
            "FPR": float(falsePostiveRate(ct)[0]),
            "TS": float(threatScore(ct)[0]),
            "TSS": float(trueSkillStatistics(ct)[0]),
            "Bias": float(bias(ct)[0]),
        }
    return pd.DataFrame.from_dict(rows, orient="index")

binary_metric_table = compute_binary_metric_table(
    {
        "AmbSoWi-ML": ambsowi_mod,
        "BRaVDA": bravda_mod,
        "Persistence (27d)": p27_mod,
    },
    omni_obs,
    threshold,
)

display(binary_metric_table.round(2))

## Visual Diagnostics for Binary Event Metrics

The visualizations in this section include:
- classification timelines
- cumulative outcome plots
- bar charts summarizing event categories

The visualizations below use **AmbSoWi-ML** against **OMNI** with the **27-day persistence** baseline.


In [ ]:
fig = create_binary_classification_plot(predicted_ambsowi, observed_data, threshold=threshold)
fig.update_layout(width=1200, height=520)
fig.show()

In [ ]:
fig = create_cumulative_binary_outcomes(predicted_ambsowi, observed_data, threshold=threshold)
fig.update_layout(width=500, height=500)
fig.show()

In [ ]:
fig = create_binary_outcome_bars(predicted_ambsowi, observed_data, baseline_data, threshold=threshold)
fig.update_layout(width=500, height=500)
fig.show()

## PDF export example

If `kaleido` is installed, Plotly figures can be exported as vector PDFs.

In [ ]:
import os

out_dir = "../results/output_pdfs"
os.makedirs(out_dir, exist_ok=True)

fig = create_binary_outcome_bars(predicted_ambsowi, observed_data, baseline_data, threshold=threshold)
fig.update_layout(width=900, height=520)

pdf_path = os.path.join(out_dir, "binary_outcome_bars_ambsowi.pdf")

# Requires kaleido:
fig.write_image(pdf_path)

print("Saved:", pdf_path)

Save all visualizations generated in this notebook as PDF files in `../results/output_pdfs/`. This requires `kaleido`.

In [ ]:
from pathlib import Path

out_dir = Path("../results/output_pdfs")
out_dir.mkdir(parents=True, exist_ok=True)

figures = {
    "binary_classification_ambsowi": create_binary_classification_plot(
        predicted_ambsowi,
        observed_data,
        threshold=threshold,
    ),
    "cumulative_binary_outcomes_ambsowi": create_cumulative_binary_outcomes(
        predicted_ambsowi,
        observed_data,
        threshold=threshold,
    ),
    "binary_outcome_bars_ambsowi": create_binary_outcome_bars(
        predicted_ambsowi,
        observed_data,
        baseline_data,
        threshold=threshold,
    ),
}

figure_sizes = {
    "binary_classification_ambsowi": (1200, 520),
    "cumulative_binary_outcomes_ambsowi": (900, 520),
    "binary_outcome_bars_ambsowi": (900, 520),
}

for name, fig in figures.items():
    width, height = figure_sizes[name]
    fig.update_layout(width=width, height=height)

    pdf_path = out_dir / f"{name}.pdf"

    # Requires kaleido:
    fig.write_image(pdf_path)

    print("Saved:", pdf_path)